Here, we will try Understanding how to use Output Parsers..
Basically, an Output Parser is simply used to format the answer being given by the llm to us, in the form of json, or csv, or any type, depending on what format we want, and what type of outputparser() we use.
THe First 4 blocks here are just the setup required. The concept of output parser starts from the 5th Row.

In [1]:
%%capture
!pip install --force-reinstall --no-cache-dir tenacity==8.2.3 --user
!pip install "ibm-watsonx-ai==1.0.8" --user
!pip install "ibm-watson-machine-learning==1.0.367" --user
!pip install "langchain-ibm==0.1.7" --user
!pip install "langchain-community==0.2.10" --user
!pip install "langchain-experimental==0.0.62" --user
!pip install "langchainhub==0.1.18" --user
!pip install "langchain==0.2.11" --user
!pip install "pypdf==4.2.0" --user
!pip install "chromadb==0.4.24" --user

In [ ]:
import os
os._exit(00)

In [1]:
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
import os
os.environ['ANONYMIZED_TELEMETRY'] = 'False'

from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM

In [2]:
model_id = 'ibm/granite-4-h-small' 

parameters = {
    GenParams.MAX_NEW_TOKENS: 256,  # this controls the maximum number of tokens in the generated output
    GenParams.TEMPERATURE: 0.2, # this randomness or creativity of the model's responses 
}

credentials = {
    "url": "https://us-south.ml.cloud.ibm.com"
    # "api_key": "your api key here"
    # uncomment above and fill in the API key when running locally
}

project_id = "skills-network"

model = ModelInference(
    model_id=model_id,
    params=parameters,
    credentials=credentials,
    project_id=project_id
)
#Adding a llama model, because something is wrong with granite ig
llama_model = ModelInference(
    model_id='meta-llama/llama-4-maverick-17b-128e-instruct-fp8',
    params=parameters,
    credentials=credentials,
    project_id=project_id
)

In [3]:
llama_llm = WatsonxLLM(model = model)
llama_llm2 = WatsonxLLM(model=llama_model) #added llama model, because granite wasn't working ig

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

In [5]:
# Create your JSON parser
json_parser = JsonOutputParser()

# Create the format instructions
format_instructions = """RESPONSE FORMAT: Return ONLY a single JSON object—no markdown, no examples, no extra keys.  It must look exactly like:
{
  "title": "movie title",
  "director": "director name",
  "year": 2000,
  "genre": "movie genre"
}

IMPORTANT: Your response must be *only* that JSON.  Do NOT include any illustrative or example JSON."""

# Create prompt template with instructions
prompt_template = PromptTemplate(
    template="""You are a JSON-only assistant.

Task: Generate info about the movie "{movie_name1}" in JSON format.

{format_instructions}
""",
    input_variables=["movie_name1"],
    partial_variables={"format_instructions": format_instructions},
)

# Create the chain
movie_chain =  prompt_template | llama_llm2 | json_parser

# Test with a movie name
movie_container = "The Matrix"
result = movie_chain.invoke({"movie_name1": movie_container})

# Print the structured result
print("Parsed result:")
print(f"Title: {result['title']}")
print(f"Director: {result['director']}")
print(f"Year: {result['year']}")
print(f"Genre: {result['genre']}")

Parsed result:
Title: The Matrix
Director: The Wachowskis
Year: 1999
Genre: Science Fiction


In [6]:
movies = ["The Matrix", "Inception", "The Godfather"]

for movie in movies:
    result = movie_chain.invoke({"movie_name1": movie})
    print(f"\n=== {movie} ===")
    print(f"Title: {result['title']}")
    print(f"Director: {result['director']}")
    print(f"Year: {result['year']}")
    print(f"Genre: {result['genre']}")


=== The Matrix ===
Title: The Matrix
Director: The Wachowskis
Year: 1999
Genre: Science Fiction

=== Inception ===
Title: Inception
Director: Christopher Nolan
Year: 2010
Genre: Action, Sci-Fi

=== The Godfather ===
Title: The Godfather
Director: Francis Ford Coppola
Year: 1972
Genre: Crime, Drama
